# Train SymGate-VDSR (v3 — refine LR floor fix)

Wires together `model_symgate_vdsr.py`, `dataloader_anchor.py`, `hann_merge.py`.

### Change from v2: refine LR floor bug (root cause of slope RMSE drift)

`CosineAnnealingWarmRestarts` applies one `eta_min` to all param groups.
With `COSINE_ETA_MIN=1e-5` and `REFINE_LR_SCALE=0.1`, the refine group
should floor at `1e-6` but was being clipped to `1e-5`.

From epoch ~25 the refine LR hit that floor and froze. Main LR kept cycling
between `1e-5` and `1e-4`. The CSPN head stopped learning while encoders kept
updating -- slope RMSE drifted from 1.057 (ep 6) to 1.146 (ep 39).

Fix: after every `scheduler.step()`, re-derive refine LR as
`max(main_lr * scale, eta_min * scale)`.

| | v2 | v3 |
|---|---|---|
| Refine LR floor | 1e-5 (same as main, wrong) | 1e-6 (eta_min * scale) |
| Refine LR post-warmup | Frozen at 1e-5 from ep ~25 | Tracks main LR x 0.1 throughout |


In [ ]:
import os, sys, json, math, time, gc, shutil, zipfile
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from dataloader_anchor import create_dataloaders, val_patch_generator
from model_symgatevdsr import (
    SymGateVDSR, SymGateTopoLoss,
    build_recommended_scheduler, branch_grad_norms,
)
from hann_merge import HannStreamMerger

# Detect environment
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IN_LOCAL = not IN_COLAB
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.backends.cudnn.benchmark = True

# =============================================================================
# LOCAL WORKSTATION (P4000)
# =============================================================================
if IN_LOCAL:
    PROJECT_DIR = Path(r'D:\Vaibhav\dem-lidar')   # <-- edit if different
    DATASET_DIR = PROJECT_DIR / 'Dataset'
    if not DATASET_DIR.exists():
        raise FileNotFoundError(
            f'Dataset not found at {DATASET_DIR}.\n'
            f'Download it and place it at {DATASET_DIR}, or update PROJECT_DIR above.'
        )
    if str(PROJECT_DIR) not in sys.path:
        sys.path.insert(0, str(PROJECT_DIR))
    print(f'Dataset found: {DATASET_DIR}')
    print(f'Scripts:       {PROJECT_DIR}')

# =============================================================================
# GOOGLE COLAB (T4)
# =============================================================================
if IN_COLAB:
    ZIP_FILENAME = 'Dataset-20260703T222224Z-3-001.zip'
    LOCAL_ZIP_PATH = Path(f'/content/{ZIP_FILENAME}')
    LOCAL_DATASET = Path('/content/Dataset')

    if not LOCAL_DATASET.exists():
        if not LOCAL_ZIP_PATH.exists():
            from google.colab import drive
            drive.mount('/content/drive')
            DRIVE_ZIP_PATH = Path(f'/content/drive/MyDrive/dem-lidar/{ZIP_FILENAME}')
            if DRIVE_ZIP_PATH.exists():
                print(f'Copying {ZIP_FILENAME} from Drive to local SSD...')
                shutil.copy(str(DRIVE_ZIP_PATH), str(LOCAL_ZIP_PATH))
                print('Copy complete.')
            else:
                raise FileNotFoundError(
                    f'Zip file not found at {DRIVE_ZIP_PATH}.\n'
                    f'Please upload {ZIP_FILENAME} to your Google Drive at that path.'
                )
        print(f'Extracting {ZIP_FILENAME}...')
        with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall('/content')
        print('Extraction done.')
    else:
        print(f'Local dataset already exists and is extracted: {LOCAL_DATASET}')

    SCRIPT_DIR = Path('/content')
    if str(SCRIPT_DIR) not in sys.path:
        sys.path.insert(0, str(SCRIPT_DIR))


In [ ]:
from pathlib import Path

# ---- Paths ------------------------------------------------------------------
if IN_LOCAL:
    BASE_DIR       = str(DATASET_DIR)
    CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints_symgate_vdsr'

if IN_COLAB:
    BASE_DIR       = str(LOCAL_DATASET)
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/checkpoints_symgate_vdsr')

TRAIN_DIRS = [
    f"{BASE_DIR}/Kl/tensors/train",
    f"{BASE_DIR}/SG/tensors/train",
]
VAL_DIRS = [
    f"{BASE_DIR}/Kl/tensors/validation_contiguous",
    f"{BASE_DIR}/SG/tensors/validation_contiguous",
]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = "symgate_vdsr"

# ---- Model ------------------------------------------------------------------
FEATURES   = 64
NUM_GROUPS = 8
CSPN_ITERS = 6

# ---- Optimiser & LR ---------------------------------------------------------
LR               = 1e-4
REFINE_LR_SCALE  = 0.1
WEIGHT_DECAY     = 5e-5
GRAD_CLIP_NORM   = 1.0
WARMUP_STEPS     = 12 * (1649 // 16) + 16

# ---- Scheduler --------------------------------------------------------------
COSINE_T0      = 40
COSINE_T_MULT  = 1
COSINE_ETA_MIN = 1e-5

# ---- Loss weights -----------------------------------------------------------
LOSS_ALPHA      = 0.5
LOSS_BETA       = 2.5
LOSS_GAMMA      = 0.5
LOSS_LAMBDA_PIN = 1.0
LOSS_LAMBDA_AUX = 0.4
PIN_BETA        = 1.0
DECAY_RADIUS    = 15.0
BUFFER_SIZE     = 3
HALO_WEIGHT     = 0.25
# ---- EMA --------------------------------------------------------------------
EMA_DECAY = 0.999

# ---- LR Restart Decay -------------------------------------------------------
LR_DECAY_ON_RESTART = 0.5

# ---- Grad norm logging ------------------------------------------------------
LOG_GRAD_EPOCHS = 20
LOG_GRAD_EVERY  = 10

# ---- Validation frequency ---------------------------------------------------
VAL_ALWAYS_EPOCHS = 20
VAL_EVERY_N       = 3

# ---- Checkpoint rotation ----------------------------------------------------
KEEP_LAST_N_CKPTS = 500

# ---- Data -------------------------------------------------------------------
BATCH_SIZE  = 16
TRAIN_CROP  = 128
VAL_CROP    = 256
VAL_OVERLAP = 192

# Environment-specific data config.
# Colab T4: 12.7 GB RAM total. After model+optimizer+CUDA cache ~1.5 GB used,
# headroom is ~11 GB. With streaming val (see validate_one_epoch), peak per
# tile is ~300 MB so a large VAL_PATCH_BATCH is safe. Reduced NUM_WORKERS
# since each worker holds a noise_cache (16 MB) + prefetch buffer.
if IN_COLAB:
    NUM_WORKERS     = 2
    VAL_PATCH_BATCH = 16   # conservative: keeps GPU batch small, CPU spike tiny
else:
    NUM_WORKERS     = 8
    VAL_PATCH_BATCH = 32

# ---- Training control -------------------------------------------------------
EPOCHS                  = 500
EARLY_STOPPING_PATIENCE = 500
BEST_METRIC_NAME        = "composite"

training_config = {
    'run_name': RUN_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'refine_lr_scale': REFINE_LR_SCALE,
    'warmup_steps': WARMUP_STEPS,
    'weight_decay': WEIGHT_DECAY,
    'grad_clip_norm': GRAD_CLIP_NORM,
    'features': FEATURES,
    'num_groups': NUM_GROUPS,
    'cspn_iters': CSPN_ITERS,
    'loss_alpha': LOSS_ALPHA,
    'loss_beta': LOSS_BETA,
    'loss_gamma': LOSS_GAMMA,
    'loss_lambda_pin': LOSS_LAMBDA_PIN,
    'loss_lambda_aux': LOSS_LAMBDA_AUX,
    'pin_beta': PIN_BETA,
    'decay_radius': DECAY_RADIUS,
    'buffer_size': BUFFER_SIZE,
    'halo_weight': HALO_WEIGHT,
    'cosine_t0': COSINE_T0,
    'cosine_t_mult': COSINE_T_MULT,
    'cosine_eta_min': COSINE_ETA_MIN,
    'log_grad_epochs': LOG_GRAD_EPOCHS,
    'log_grad_every': LOG_GRAD_EVERY,
    'val_always_epochs': VAL_ALWAYS_EPOCHS,
    'val_every_n': VAL_EVERY_N,
    'keep_last_n_ckpts': KEEP_LAST_N_CKPTS,
    'train_crop': TRAIN_CROP,
    'val_crop': VAL_CROP,
    'val_overlap': VAL_OVERLAP,
    'val_patch_batch': VAL_PATCH_BATCH,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'best_metric_name': BEST_METRIC_NAME,
    'train_dirs': TRAIN_DIRS,
    'val_dirs': VAL_DIRS,
}
print(json.dumps(training_config, indent=2))


In [ ]:
# ---- Metric helpers ---------------------------------------------------------
def compute_rmse(pred, gt):
    return torch.sqrt(torch.mean((pred - gt) ** 2))

def compute_psnr(pred, gt, eps=1e-8):
    mse = torch.mean((pred - gt) ** 2)
    data_range = torch.clamp(gt.max() - gt.min(), min=eps)
    if mse <= eps:
        return torch.tensor(float('inf'), device=pred.device)
    return 20.0 * torch.log10(data_range) - 10.0 * torch.log10(torch.clamp(mse, min=eps))

def safe_conv(tensor, kernel):
    padded = F.pad(tensor, (1, 1, 1, 1), mode='replicate')
    return F.conv2d(padded, kernel)

sobel_x = torch.tensor(
    [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

sobel_y = torch.tensor(
    [[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

laplacian = torch.tensor(
    [[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32
).view(1, 1, 3, 3)

def get_warmup_lr(step, base_lr, warmup_steps):
    if step >= warmup_steps:
        return base_lr
    return base_lr * (step + 1) / warmup_steps

# ---- Checkpoint save with rotation and I/O guard ----------------------------
def save_checkpoint(
    epoch, model, optimizer, scheduler,
    best_metric, best_epoch, train_history, checkpoint_dir, training_config,
    is_best=False, keep_last_n=5,
):
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_metric': best_metric,
        'best_epoch': best_epoch,
        'best_metric_name': training_config.get('best_metric_name', 'composite'),
        'train_history': train_history,
        'training_config': training_config,
        'torch_version': torch.__version__,
    }

    filename = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}.pt"

    # Wrapped save: an I/O hiccup should log a warning, not kill the run.
    try:
        torch.save(ckpt, filename)
        meta_path = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}_meta.json"
        with open(meta_path, 'w', encoding='utf-8') as fh:
            json.dump({
                'epoch': epoch,
                'best_metric': best_metric,
                'best_epoch': best_epoch,
                'training_config': training_config,
                'torch_version': torch.__version__,
            }, fh, indent=2)
    except Exception as e:
        tqdm.write(f"WARNING: checkpoint save failed at epoch {epoch}: {e}")
        return None

    if is_best:
        best_path = checkpoint_dir / f"{RUN_NAME}_best.pt"
        try:
            torch.save(ckpt, best_path)
        except Exception as e:
            tqdm.write(f"WARNING: best checkpoint save failed at epoch {epoch}: {e}")

    # Checkpoint rotation: delete old per-epoch files beyond keep_last_n.
    # The _best.pt file is never touched by rotation.
    if keep_last_n is not None and keep_last_n > 0:
        import glob
        pattern = str(checkpoint_dir / f"{RUN_NAME}_epoch_*.pt")
        epoch_ckpts = sorted(glob.glob(pattern))
        while len(epoch_ckpts) > keep_last_n:
            oldest = epoch_ckpts.pop(0)
            try:
                import os as _os
                _os.remove(oldest)
                meta = oldest.replace('.pt', '_meta.json')
                if _os.path.exists(meta):
                    _os.remove(meta)
            except Exception as e:
                tqdm.write(f"WARNING: could not delete old checkpoint {oldest}: {e}")

    return filename

# ---- train_one_epoch --------------------------------------------------------
# log_grad_norms: passed in per-epoch from main() -- True only for the first
# LOG_GRAD_EPOCHS epochs and every LOG_GRAD_EVERY epochs after that.
# Each branch_grad_norms() call does one GPU->CPU sync per parameter tensor;
# over ~103 batches * 500 epochs that is a material serialisation cost.
# When False, the gn_ keys are set to -1.0 in the returned dict (sentinel,
# not NaN) so the history row shape stays consistent for downstream analysis.
# -----------------------------------------------------------------------------
# ---- Metric helpers ---------------------------------------------------------

class EMAModel:
    """Exponential Moving Average shadow model.
    
    Maintains shadow weights updated via EMA each training step.
    Use apply_shadow() before eval to evaluate on averaged weights;
    restore() after to put raw weights back.
    """

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self.device = next(model.parameters()).device
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone().to(self.device)
        for name, buf in model.named_buffers():
            self.backup[name] = buf.data.clone().to(self.device)

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (
                    self.decay * self.shadow[name]
                    + (1.0 - self.decay) * param.data.to(self.device)
                )

    def apply_shadow(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name].to(param.device))
        for name, buf in model.named_buffers():
            self.backup[name] = buf.data.clone()
            if name in self.shadow:
                buf.data.copy_(self.shadow[name].to(buf.device))

    def restore(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name].to(param.device))
        for name, buf in model.named_buffers():
            buf.data.copy_(self.backup[name].to(buf.device))


def compute_rmse(pred, gt):
    return torch.sqrt(torch.mean((pred - gt) ** 2))

def compute_psnr(pred, gt, eps=1e-8):
    mse = torch.mean((pred - gt) ** 2)
    data_range = torch.clamp(gt.max() - gt.min(), min=eps)
    if mse <= eps:
        return torch.tensor(float('inf'), device=pred.device)
    return 20.0  *torch.log10(data_range) - 10.0*  torch.log10(torch.clamp(mse, min=eps))

def safe_conv(tensor, kernel):
    padded = F.pad(tensor, (1, 1, 1, 1), mode='replicate')
    return F.conv2d(padded, kernel)

sobel_x = torch.tensor(
    [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

sobel_y = torch.tensor(
    [[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

laplacian = torch.tensor(
    [[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32
).view(1, 1, 3, 3)

def get_warmup_lr(step, base_lr, warmup_steps):
    if step >= warmup_steps:
        return base_lr
    return base_lr * (step + 1) / warmup_steps

# ---- Checkpoint save with rotation and I/O guard ----------------------------

def save_checkpoint(
    epoch, model, optimizer, scheduler,
    best_metric, best_epoch, train_history, checkpoint_dir, training_config,
    is_best=False, keep_last_n=5,
):
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_metric': best_metric,
        'best_epoch': best_epoch,
        'best_metric_name': training_config.get('best_metric_name', 'composite'),
        'train_history': train_history,
        'training_config': training_config,
        'torch_version': torch.__version__,
    }
    filename = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}.pt"
    # Wrapped save: an I/O hiccup should log a warning, not kill the run.
    try:
        torch.save(ckpt, filename)
        meta_path = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}_meta.json"
        with open(meta_path, 'w', encoding='utf-8') as fh:
            json.dump({
                'epoch': epoch,
                'best_metric': best_metric,
                'best_epoch': best_epoch,
                'training_config': training_config,
                'torch_version': torch.__version__,
            }, fh, indent=2)
    except Exception as e:
        tqdm.write(f"WARNING: checkpoint save failed at epoch {epoch}: {e}")
        return None
    if is_best:
        best_path = checkpoint_dir / f"{RUN_NAME}_best.pt"
        try:
            torch.save(ckpt, best_path)
        except Exception as e:
            tqdm.write(f"WARNING: best checkpoint save failed at epoch {epoch}: {e}")
    # Checkpoint rotation: delete old per-epoch files beyond keep_last_n.
    # The _[best.pt](http://best.pt) file is never touched by rotation.
    if keep_last_n is not None and keep_last_n > 0:
        import glob
        pattern = str(checkpoint_dir / f"{RUN_NAME}_epoch_*.pt")
        epoch_ckpts = sorted(glob.glob(pattern))
        while len(epoch_ckpts) > keep_last_n:
            oldest = epoch_ckpts.pop(0)
            try:
                import os as _os
                _os.remove(oldest)
                meta = oldest.replace('.pt', '_meta.json')
                if _os.path.exists(meta):
                    _os.remove(meta)
            except Exception as e:
                tqdm.write(f"WARNING: could not delete old checkpoint {oldest}: {e}")
    return filename

# ---- train_one_epoch --------------------------------------------------------

# log_grad_norms: passed in per-epoch from main() -- True only for the first
# LOG_GRAD_EPOCHS epochs and every LOG_GRAD_EVERY epochs after that.
# Each branch_grad_norms() call does one GPU->CPU sync per parameter tensor;
# over ~103 batches * 500 epochs that is a material serialisation cost.
# When False, the gn_ keys are set to -1.0 in the returned dict (sentinel,
# not NaN) so the history row shape stays consistent for downstream analysis.
# -----------------------------------------------------------------------------

def train_one_epoch(
    model, criterion, optimizer, scheduler, train_loader, device,
    grad_clip_norm=1.0, epoch=0, warmup_steps=0, base_lr=1e-4,
    log_grad_norms=False, ema=None,
):
    model.train()
    running = {
        'total': 0.0, 'base': 0.0, 'slope': 0.0, 'curve': 0.0,
        'pin': 0.0, 'aux': 0.0,
        'mae': 0.0, 'rmse': 0.0,
        'gn_dense': 0.0, 'gn_sparse': 0.0, 'gn_fusion': 0.0,
        'gn_refine': 0.0, 'gn_head': 0.0,
    }
    n_batches = 0
    global_step_offset = (epoch - 1) * len(train_loader)
    pbar = tqdm(train_loader, desc='Train', leave=False, dynamic_ncols=True)
    for batch_idx, batch in enumerate(pbar):
        dem_bic     = batch['dem_bic'].to(device, non_blocking=True, dtype=torch.float32)
        lidar_delta = batch['lidar_delta'].to(device, non_blocking=True, dtype=torch.float32)
        mask        = batch['mask'].to(device, non_blocking=True, dtype=torch.float32)
        dist_map    = batch['dist_map'].to(device, non_blocking=True, dtype=torch.float32)
        gt_dem      = batch['gt_dem'].to(device, non_blocking=True, dtype=torch.float32)
        norm_mean   = batch['norm_patch_mean'].to(device, non_blocking=True, dtype=torch.float32)
        lidar_raw   = dem_bic + lidar_delta
        # Linear LR warmup (step-based, main param group only)
        global_step = global_step_offset + batch_idx
        if global_step < warmup_steps:
            warm_lr = get_warmup_lr(global_step, base_lr, warmup_steps)
            optimizer.param_groups[0]['lr'] = warm_lr
            # refine group tracks at its fixed ratio
            optimizer.param_groups[1]['lr'] = max(warm_lr  *REFINE_LR_SCALE, COSINE_ETA_MIN*  REFINE_LR_SCALE)
        optimizer.zero_grad(set_to_none=True)
        outputs   = model(dem_bic, lidar_delta, mask, dist_map, patch_mean=norm_mean)
        loss_dict = criterion(outputs, gt_dem, lidar_raw, lidar_delta, mask, dist_map)
        if not torch.isfinite(loss_dict['total']):
            tqdm.write(f"WARNING: non-finite loss at epoch {epoch}, batch {batch_idx} -- skipping")
            optimizer.zero_grad(set_to_none=True)
            continue
        loss_dict['total'].backward()
        # Grad norms: gated to avoid per-tensor GPU->CPU sync on every batch.
        if log_grad_norms:
            gn = branch_grad_norms(model)
            running['gn_dense']  += gn['dense_encoder']
            running['gn_sparse'] += gn['sparse_encoder']
            running['gn_fusion'] += gn['fusion']
            running['gn_refine'] += gn['refine']
            running['gn_head']   += gn['head']
        if grad_clip_norm is not None and grad_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()
        # Update EMA after optimizer step
        if ema is not None:
            ema.update(model)
        # CosineAnnealingWarmRestarts steps per batch after warmup
        if global_step >= warmup_steps:
            epoch_continuous = epoch - 1 + batch_idx / len(train_loader)
            # Detect LR restart: T_cur resets to 0 when a new cycle begins.
            # Multiply peak LR by decay factor at each restart.
            prev_t_cur = scheduler.T_cur
            scheduler.step(epoch_continuous)
            curr_t_cur = scheduler.T_cur
            # When T_cur drops from >0 to 0, a restart just occurred.
            if prev_t_cur > 0 and curr_t_cur == 0:
                # Mutate the scheduler's own base_lrs, not param_groups[i]['lr'] --
                # CAWR recomputes lr from self.base_lrs on every .step() call, so a
                # direct param_groups[i]['lr'] *= decay gets silently overwritten
                # on the very next batch's .step().
                scheduler.base_lrs = [b * LR_DECAY_ON_RESTART for b in scheduler.base_lrs]
            # Re-derive refine LR: CAWR applies one eta_min to all param groups.
            # Refine group needs its own lower floor (eta_min * scale = 1e-6),
            # not the shared 1e-5. Without this, refine hits floor at ep ~25 and
            # freezes while main LR keeps cycling -- causes slope RMSE to drift.
            main_lr = optimizer.param_groups[0]['lr']
            optimizer.param_groups[1]['lr'] = max(
                main_lr * REFINE_LR_SCALE,
                COSINE_ETA_MIN * REFINE_LR_SCALE
            )
        with torch.no_grad():
            pred_dem   = outputs['dem_pred']
            batch_mae  = F.l1_loss(pred_dem, gt_dem)
            batch_rmse = compute_rmse(pred_dem, gt_dem)
        running['total'] += float(loss_dict['total'].item())
        running['base']  += float(loss_dict['base'].item())
        running['slope'] += float(loss_dict['slope'].item())
        running['curve'] += float(loss_dict['curve'].item())
        running['pin']   += float(loss_dict['pin'].item())
        running['aux']   += float(loss_dict['aux'].item())
        running['mae']   += float(batch_mae.item())
        running['rmse']  += float(batch_rmse.item())
        n_batches += 1
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.4f}",
            'pin':  f"{loss_dict['pin'].item():.4f}",
            'aux':  f"{loss_dict['aux'].item():.4f}",
            'mae':  f"{batch_mae.item():.4f}",
            'lr':   f"{optimizer.param_groups[0]['lr']:.2e}",
        })
    n = max(n_batches, 1)
    for k in running:
        running[k] /= n
    # If grad norms were not logged this epoch, use sentinel -1.0
    if not log_grad_norms:
        for k in ('gn_dense', 'gn_sparse', 'gn_fusion', 'gn_refine', 'gn_head'):
            running[k] = -1.0
    return running


In [ ]:
# ---- validate_one_epoch -----------------------------------------------------
# Memory-safe streaming validation using val_patch_generator().
# -----------------------------------------------------------------------------

@torch.inference_mode()
def validate_one_epoch(model, criterion, val_dataset, device, val_patch_batch=16, ema=None):
    # Use EMA shadow weights for validation if provided.
    ema_used = False
    if ema is not None:
        ema.apply_shadow(model)
        ema_used = True

    model.eval()

    # Ensure Sobel/Laplacian kernels are on the correct device
    sobel_x_dev   = sobel_x.to(device)
    sobel_y_dev   = sobel_y.to(device)
    laplacian_dev = laplacian.to(device)

    total_loss       = 0.0
    total_mae        = 0.0
    total_rmse       = 0.0
    total_psnr       = 0.0
    total_slope_rmse = 0.0
    total_curve_rmse = 0.0
    total_anchor_mae = 0.0
    total_aux_mae    = 0.0
    n_samples        = 0

    outer = tqdm(
        range(len(val_dataset)),
        desc='Validate files', leave=False, dynamic_ncols=True,
    )

    for file_idx in outer:
        # ------------------------------------------------------------------
        # 1. Load one file -- raw numpy arrays, no patch stacking
        # ------------------------------------------------------------------
        file_item = val_dataset[file_idx]
        canvas_shape   = file_item['canvas_shape']
        original_shape = file_item['original_shape']
        pad            = file_item['pad']
        gt_canvas_np   = file_item['gt_canvas_full']   # (H_orig, W_orig) float32
        n_patches       = len(file_item['coords_list'])
        n_patch_batches = math.ceil(n_patches / val_patch_batch)

        merger = HannStreamMerger(
            canvas_shape=canvas_shape,
            patch_size=256,
            device='cpu',
            pad=pad,
            original_shape=original_shape,
        )

        running_val_loss    = 0.0
        file_anchor_mae_sum = 0.0
        file_aux_mae_sum    = 0.0
        file_mask_sum       = 0.0
        n_val_batches       = 0

        inner = tqdm(
            val_patch_generator(file_item, patch_batch_size=val_patch_batch),
            total=n_patch_batches,
            desc=f'File {file_idx+1}/{len(val_dataset)}',
            leave=False, dynamic_ncols=True,
        )

        # ------------------------------------------------------------------
        # 2. Inner loop: one mini-batch at a time (O(1) extra RAM)
        # ------------------------------------------------------------------
        for pb in inner:
            dem_bic_b     = pb['dem_bic'].to(device, non_blocking=True)
            lidar_delta_b = pb['lidar_delta'].to(device, non_blocking=True)
            mask_b        = pb['mask'].to(device, non_blocking=True)
            dist_map_b    = pb['dist_map'].to(device, non_blocking=True)
            gt_dem_b      = pb['gt_dem'].to(device, non_blocking=True)
            
            # Pass the NORMALIZED mean to the model for the confidence head
            norm_mean_b   = pb['norm_patch_mean'].to(device, non_blocking=True)
            
            lidar_raw_b   = dem_bic_b + lidar_delta_b
            
            # Model now takes patch_mean (normalized) for the confidence head
            outputs  = model(dem_bic_b, lidar_delta_b, mask_b, dist_map_b, patch_mean=norm_mean_b)
            
            pred_dem = outputs['dem_pred']
            r_aux    = outputs['r_anchor_aux']
            loss_dict = criterion(
                outputs, gt_dem_b, lidar_raw_b, lidar_delta_b, mask_b, dist_map_b
            )

            running_val_loss += float(loss_dict['total'].item())
            mask_sum_b = float(mask_b.sum().item())

            if mask_sum_b > 0:
                file_anchor_mae_sum += float(
                    (mask_b * torch.abs(pred_dem - lidar_raw_b)).sum().item()
                )
                file_aux_mae_sum += float(
                    (mask_b * torch.abs(r_aux - lidar_delta_b)).sum().item()
                )
                file_mask_sum += mask_sum_b

            # Move pred to CPU for Hann merge
            pred_cpu = pred_dem.detach().cpu()

            # Delete GPU tensors immediately
            del dem_bic_b, lidar_delta_b, mask_b, dist_map_b, gt_dem_b, norm_mean_b
            del lidar_raw_b, outputs, pred_dem, r_aux, loss_dict
            torch.cuda.empty_cache()

            # IMPORTANT: Pass the RAW patch_mean to the merger for correct un-centering
            merger.add_batch(
                pred_cpu,
                pb['patch_mean'], # Raw elevation mean
                pb['coords'].cpu(),
            )
            del pred_cpu
            n_val_batches += 1
            inner.set_postfix({'mini': f'{n_val_batches}/{n_patch_batches}'})

        # ------------------------------------------------------------------
        # 3. Free file data (raw numpy arrays) before computing metrics
        # ------------------------------------------------------------------
        del file_item
        val_loss = running_val_loss / max(n_val_batches, 1)
        val_anchor_mae = file_anchor_mae_sum / max(file_mask_sum, 1.0)
        val_aux_mae    = file_aux_mae_sum    / max(file_mask_sum, 1.0)

        # ------------------------------------------------------------------
        # 4. Stitch and compute full-DEM metrics on GPU
        # ------------------------------------------------------------------
        merged_pred = merger.get_final_dem(as_tensor=True).unsqueeze(0).unsqueeze(0)
        del merger
        gt_canvas = torch.from_numpy(gt_canvas_np).unsqueeze(0).unsqueeze(0)
        del gt_canvas_np

        merged_pred_dev = merged_pred.to(device)
        gt_canvas_dev   = gt_canvas.to(device)
        del merged_pred, gt_canvas

        mae  = F.l1_loss(merged_pred_dev, gt_canvas_dev)
        rmse = compute_rmse(merged_pred_dev, gt_canvas_dev)
        psnr = compute_psnr(merged_pred_dev, gt_canvas_dev)

        pred_dx    = safe_conv(merged_pred_dev, sobel_x_dev)
        pred_dy    = safe_conv(merged_pred_dev, sobel_y_dev)
        gt_dx      = safe_conv(gt_canvas_dev,   sobel_x_dev)
        gt_dy      = safe_conv(gt_canvas_dev,   sobel_y_dev)

        slope_rmse = compute_rmse(
            torch.sqrt(pred_dx**2 + pred_dy**2),
            torch.sqrt(gt_dx**2   + gt_dy**2),
        )

        curve_rmse = compute_rmse(
            safe_conv(merged_pred_dev, laplacian_dev),
            safe_conv(gt_canvas_dev,   laplacian_dev),
        )

        del merged_pred_dev, gt_canvas_dev
        del pred_dx, pred_dy, gt_dx, gt_dy
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        # ------------------------------------------------------------------
        # 5. Accumulate
        # ------------------------------------------------------------------
        total_loss       += val_loss
        total_mae        += float(mae.item())
        total_rmse       += float(rmse.item())
        total_psnr       += float(psnr.item())
        total_slope_rmse += float(slope_rmse.item())
        total_curve_rmse += float(curve_rmse.item())
        total_anchor_mae += val_anchor_mae
        total_aux_mae    += val_aux_mae
        n_samples        += 1

        outer.set_postfix({
            'mae':   f'{mae.item():.3f}',
            'rmse':  f'{rmse.item():.3f}',
            'slope': f'{slope_rmse.item():.3f}',
            'aux':   f'{val_aux_mae:.4f}',
        })

    if ema_used:
        ema.restore(model)

    return {
        'val_total':      total_loss       / max(n_samples, 1),
        'val_mae':        total_mae        / max(n_samples, 1),
        'val_rmse':       total_rmse       / max(n_samples, 1),
        'val_psnr':       total_psnr       / max(n_samples, 1),
        'val_slope_rmse': total_slope_rmse / max(n_samples, 1),
        'val_curve_rmse': total_curve_rmse / max(n_samples, 1),
        'val_anchor_mae': total_anchor_mae / max(n_samples, 1),
        'val_aux_mae':    total_aux_mae    / max(n_samples, 1),
    }

In [ ]:
def should_validate(epoch, val_always_epochs, val_every_n, cosine_t0):
    if epoch <= val_always_epochs:
        return True
    if epoch % cosine_t0 == 0:  # always validate at restart boundaries
        return True
    if epoch % val_every_n == 0:
        return True
    return False

def main():
    train_loader, val_dataset = create_dataloaders(
        TRAIN_DIRS, VAL_DIRS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        prefetch_factor=4 if NUM_WORKERS > 0 else None,
        pin_memory=True,
    )
    
    model = SymGateVDSR(features=FEATURES, num_groups=NUM_GROUPS, cspn_iters=CSPN_ITERS).to(device)
    
    criterion = SymGateTopoLoss(
        alpha        = LOSS_ALPHA,
        beta         = LOSS_BETA,
        gamma        = LOSS_GAMMA,
        lambda_pin   = LOSS_LAMBDA_PIN,
        lambda_aux   = LOSS_LAMBDA_AUX,
        # lambda_conf uses default 0.1 if not specified in your config
        pin_beta     = PIN_BETA,
        decay_radius = DECAY_RADIUS,
        buffer_size  = BUFFER_SIZE,
        halo_weight  = HALO_WEIGHT,
    ).to(device)

    ema = EMAModel(model, decay=EMA_DECAY)

    encoder_params = (
        list(model.dense_encoder.parameters()) +
        list(model.sparse_encoder.parameters()) +
        list(model.fusion1.parameters()) +
        list(model.fusion2.parameters()) +
        list(model.fusion3.parameters()) +
        list(model.head.parameters())
    )
    refine_params = list(model.refine.parameters())

    optimizer = torch.optim.Adam([
        {'params': encoder_params, 'lr': LR},
        {'params': refine_params,  'lr': LR * REFINE_LR_SCALE},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = build_recommended_scheduler(
        optimizer,
        warm_restart_epochs=COSINE_T0,
        min_lr_floor=COSINE_ETA_MIN,
    )

    RESUME_CHECKPOINT = None
    start_epoch    = 1
    best_metric    = float('inf')
    best_composite = float('inf')
    best_epoch     = 0
    stale_epochs   = 0
    train_history  = []
    last_val_stats = None

    if RESUME_CHECKPOINT is not None:
        ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if ckpt.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if ema is not None:
            # Re-seed the shadow FROM the just-loaded checkpoint weights --
            # ema.shadow currently holds the pre-load random init (EMAModel was
            # constructed before this resume block ran). Calling apply_shadow()
            # here would overwrite the resumed weights with that stale random
            # init instead. Reseeding lets both `model` and `ema.shadow` start
            # this resumed run from the same, correct point.
            for name, param in model.named_parameters():
                if param.requires_grad:
                    ema.shadow[name] = param.data.clone().to(ema.device)
        start_epoch    = ckpt['epoch'] + 1
        best_metric    = ckpt.get('best_metric', float('inf'))
        best_composite = ckpt.get('best_metric', float('inf'))
        best_epoch     = ckpt.get('best_epoch', 0)
        train_history  = ckpt.get('train_history', [])
        print(f'Resumed from epoch {ckpt["epoch"]} | best_composite={best_composite:.4f} @ ep {best_epoch}')
    else:
        print('Starting from scratch.')

    print(f'Train files: {len(train_loader.dataset)}')
    print(f'Val files:   {len(val_dataset)}')

    epoch_bar = tqdm(range(start_epoch, EPOCHS + 1), desc='Epochs', dynamic_ncols=True)

    for epoch in epoch_bar:
        t0 = time.time()
        log_norms_this_epoch = (epoch <= LOG_GRAD_EPOCHS or epoch % LOG_GRAD_EVERY == 0)

        train_stats = train_one_epoch(
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            train_loader=train_loader,
            device=device,
            grad_clip_norm=GRAD_CLIP_NORM,
            epoch=epoch,
            warmup_steps=WARMUP_STEPS,
            base_lr=LR,
            log_grad_norms=log_norms_this_epoch,
            ema=ema,
        )

        run_val = should_validate(epoch, VAL_ALWAYS_EPOCHS, VAL_EVERY_N, COSINE_T0)

        if run_val:
            val_stats = validate_one_epoch(
                model=model,
                criterion=criterion,
                val_dataset=val_dataset,
                device=device,
                val_patch_batch=VAL_PATCH_BATCH,
                ema=ema,
            )
            last_val_stats = val_stats
            composite = val_stats['val_slope_rmse'] + val_stats['val_aux_mae']
        else:
            val_stats = last_val_stats
            composite = None

        lr_now  = optimizer.param_groups[0]['lr']
        lr_refine = optimizer.param_groups[1]['lr']
        elapsed = time.time() - t0

        row = {
            'epoch':          epoch,
            'lr':             lr_now,
            'lr_refine':      lr_refine,
            'val_run':        run_val,
            'train_total':    train_stats['total'],
            'train_base':     train_stats['base'],
            'train_slope':    train_stats['slope'],
            'train_curve':    train_stats['curve'],
            'train_pin':      train_stats['pin'],
            'train_aux':      train_stats['aux'],
            'train_mae':      train_stats['mae'],
            'train_rmse':     train_stats['rmse'],
            'gn_dense':       train_stats['gn_dense'],
            'gn_sparse':      train_stats['gn_sparse'],
            'gn_fusion':      train_stats['gn_fusion'],
            'gn_refine':      train_stats['gn_refine'],
            'gn_head':        train_stats['gn_head'],
            'val_total':      val_stats['val_total']      if val_stats else None,
            'val_mae':        val_stats['val_mae']        if val_stats else None,
            'val_rmse':       val_stats['val_rmse']       if val_stats else None,
            'val_psnr':       val_stats['val_psnr']       if val_stats else None,
            'val_slope_rmse': val_stats['val_slope_rmse'] if val_stats else None,
            'val_curve_rmse': val_stats['val_curve_rmse'] if val_stats else None,
            'val_anchor_mae': val_stats['val_anchor_mae'] if val_stats else None,
            'val_aux_mae':    val_stats['val_aux_mae']    if val_stats else None,
            'val_composite':  composite,
            'time_sec':       elapsed,
        }
        train_history.append(row)

        is_best = False
        if composite is not None:
            is_best = composite < best_composite
            if is_best:
                best_composite = composite
                best_metric    = composite
                best_epoch     = epoch
                stale_epochs   = 0
            else:
                stale_epochs  += 1

        if ema is not None:
            ema.apply_shadow(model)
        
        save_checkpoint(
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_metric=best_metric,
            best_epoch=best_epoch,
            train_history=train_history,
            checkpoint_dir=CHECKPOINT_DIR,
            training_config=training_config,
            is_best=is_best,
            keep_last_n=KEEP_LAST_N_CKPTS,
        )

        if ema is not None:
            ema.restore(model)

        epoch_bar.set_postfix({
            'train': f"{train_stats['total']:.4f}",
            'slope': f"{val_stats['val_slope_rmse']:.4f}" if val_stats else '---',
            'aux':   f"{val_stats['val_aux_mae']:.4f}"    if val_stats else '---',
            'best':  f"{best_composite:.4f}",
            'lr':    f"{lr_now:.2e}",
        })

        gn_str = ''
        if log_norms_this_epoch:
            gn_str = (
                f"GN dense={train_stats['gn_dense']:.3f} "
                f"sparse={train_stats['gn_sparse']:.3f} "
                f"fusion={train_stats['gn_fusion']:.3f} "
                f"refine={train_stats['gn_refine']:.3f} | "
            )

        val_str = (
            f"ValMAE {val_stats['val_mae']:.4f} | "
            f"SlopeRMSE {val_stats['val_slope_rmse']:.4f} | "
            f"AuxMAE {val_stats['val_aux_mae']:.4f} | "
            f"AnchorMAE(sanity) {val_stats['val_anchor_mae']:.6f} | "
            f"Composite {composite:.4f} | "
        ) if run_val else f"(val skipped, last composite {best_composite:.4f}) | "

        tqdm.write(
            f"Epoch {epoch:03d} | "
            f"TLoss {train_stats['total']:.4f} "
            f"(B={train_stats['base']:.3f} S={train_stats['slope']:.3f} "
            f"C={train_stats['curve']:.3f} P={train_stats['pin']:.3f} A={train_stats['aux']:.3f}) | "
            + gn_str +
            val_str +
            f"LR {lr_now:.2e} refine={lr_refine:.2e} | "
            f"Best {best_composite:.4f} @ ep {best_epoch} | "
            f"time {elapsed:.1f}s"
        )

        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            tqdm.write(f"Early stopping triggered at epoch {epoch}.")
            break

    epoch_bar.close()
    print('Training finished.')

In [ ]:
main()